# Pipelines and ML Workflows
## Overview
This lab focuses on the preprocessing stage of the ML workflow. You will apply each step to a dataset of your choice from the suggested list below, with the goal of producing a clean, properly preprocessed features ready for modeling. The one concept we will walk through in detail before you begin is pipelines, a scikit-learn tool that keeps your preprocessing/modeling clean and prevents data leakage.

The steps you will follow:

- Understand the big picture
- Get the data and do a preliminary exploration
- Create a representative test set and lock it away
- Explore the training data to gain insights
- Consider feature combinations
- Prepare train data for ML algorithms (Build your preprocessing pipeline!)


#### If time permits, you can go on to steps 7 - 10 f the ML Workflow:

- Try out different candidate models and estimate accuracy metrics with CV
- Go back to the drawing board if needed
- Select a model and fine-tune it
- Evaluate model on test set

## Step 1: Understand the Big Picture
### Question 1
Before touching the data, answer the following:

- What is the response variable? Is this a regression or classification problem? Fatality, classification
- What performance metric will you use, and why is it appropriate? Accuracy for a classification task
- What is a sensible baseline to beat (e.g., predicting the mean, a simple rule, a published benchmark)?
- Are there any domain-specific constraints on errors? Is over-prediction or under-prediction more costly?

In [7]:
import pandas as pd
import os
import matplotlib.pyplot as plt

# File path 
path = os.path.join('data', 'GSAF5.xls')

# Read in data
df = pd.read_excel(path, engine = 'xlrd')

In [11]:
pd.set_option('display.max_columns', None)
df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,24th May,2026.0,Unprovoked,Australia,Queensland,Kennedy Shoal,Spearfishing,Michael Jensz,M,39,Bite wounds to the head,Y,1100hrs,Undetermined Bull shark most likely,Simon de Marchi: 9 News: 7 News: ABC News:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,17th May,2026.0,Questionable,USA,Maryland,Assateague State Park,Surfing,Brendan Oster,M,?,Bite wound to the hand,N,?,Blue fish bite most probable,Keithe Cowley: Delmarva Now:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,16th May,2026.0,Unprovoked,Australia,Western Australia,Horseshoe Reef Rottnest Island,Skindiving,Steven Mattaboni,M,38,Severe injuries both legs,Y,1000hrs,Great White Shark,ABC News: 9 News: 7 News: Simon De Marchi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,14th April,2026.0,UNprovoked,Maldives,Gaafu Alif Atoll,Kooddoo,Swimming,Not stated - on honeymoon,M,?,Leg strpped off flesh later amputated in hospital,N,?,Unknown,The U.S. Sun: Simon De Marchi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3rd April,2026.0,Unprovoked,Australia,South Australia,Middleton Beach Fleurieu Peninsula Adelaide,Surfing,Oliver Tokic-Bensley,M,16,Bite wound to R ankle,N,?,Bronze Whaler,ABC News: The Guardian: Andrew Currie and Bob...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
df.describe()

,Year,original order
count,7088.000000,6799.000000
mean,1936.208521,3401.152081
std,270.400387,1963.076319
min,0.000000,2.000000
25%,1948.000000,1701.500000
50%,1987.000000,3401.000000
75%,2010.000000,5100.500000
max,2026.000000,6802.000000


In [ ]:
# Create representative test set and lock it away
from sklearn.model_selection import train_test_split

response_var = 'Fatal Y/N'   # replace

X = df.drop(columns=response_var)
y = df[response_var]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training   : {X_train.shape[0]} rows')
print(f'Test       : {X_test.shape[0]} rows')

Training   : 5672 rows
Test       : 1418 rows


In [13]:
# Explore the training data to gain insights
X_train.columns

Index(['Date', 'Year', 'Type', 'Country', 'State', 'Location', 'Activity',
       'Name', 'Sex', 'Age', 'Injury', 'Time', 'Species ', 'Source', 'pdf',
       'href formula', 'href', 'Case Number', 'Case Number.1',
       'original order', 'Unnamed: 21', 'Unnamed: 22'],
      dtype='object')

In [15]:
X_train.dtypes

Date               object
Year              float64
Type               object
Country            object
State              object
Location           object
Activity           object
Name               object
Sex                object
Age                object
Injury             object
Time               object
Species            object
Source             object
pdf                object
href formula       object
href               object
Case Number        object
Case Number.1      object
original order    float64
Unnamed: 21        object
Unnamed: 22        object
dtype: object

In [17]:
X_train.isnull().sum() * 100/ len(X_train)

Date               0.000000
Year               0.035261
Type               0.211566
Country            0.669958
State              6.840621
Location           8.110014
Activity           8.180536
Name               3.102962
Sex                8.021862
Age               42.383639
Injury             0.476023
Time              50.229196
Species           44.040903
Source             0.246827
pdf                4.072638
href formula       4.160790
href               4.125529
Case Number        4.090268
Case Number.1      4.090268
original order     4.072638
Unnamed: 21       99.982370
Unnamed: 22       99.964739
dtype: float64

In [ ]:
# Build processing pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

cat_features = ['Date', 'Year', 'Type', 'Country', 'State', 'Location', 'Activity', 'Injury', 'Species']  # fill in (empty list if none)

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipeline, cat_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f'Processed training shape: {X_train_processed.shape}')
print(f'Processed test shape    : {X_test_processed.shape}')


TypeError: Encoders require their input argument must be uniformly strings or numbers. Got ['datetime', 'int', 'str']